# Visualize the Equity Style Box

This notebook automatically reads your portfolio from `my_portfolio.py`, fetches asset data from yfinance, classifies each equity/ETF holding by **market-cap** (Large / Mid / Small) and **style** (Value / Blend / Growth), and renders a weighted Equity Style Box.

Non-equity assets (e.g. forex, bonds) are excluded from the style box and their weight is noted separately.

In [ ]:
import sys
import os
import importlib
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

sys.path.insert(0, os.path.abspath('..'))
from my_portfolio import *

In [ ]:
# ---------------------------------------------------------------------------
# 2. Helper: classify a single ticker into (cap_row, style_col)
#
#    cap_row  : 0 = Large, 1 = Mid, 2 = Small
#    style_col: 0 = Value, 1 = Blend, 2 = Growth
#
# Strategy
# --------
# A) If yfinance returns Morningstar-style category strings we parse them.
# B) For individual stocks we use marketCap + trailingPE / priceToBook.
# C) For ETFs/funds without a clear category we fall back to the fund name.
# D) Non-equity assets (quoteType CURRENCY, FUTURE, etc.) return None.
# ---------------------------------------------------------------------------

# Market-cap thresholds (USD)
LARGE_CAP_THRESHOLD = 10e9   # > 10 B  → Large
MID_CAP_THRESHOLD   =  2e9   # 2-10 B  → Mid
                              # < 2 B   → Small

# Keywords that appear in Morningstar / yfinance category strings
CAP_KEYWORDS = {
    0: ['large', 'mega', 'big'],
    1: ['mid'],
    2: ['small', 'micro', 'nano'],
}
STYLE_KEYWORDS = {
    0: ['value'],
    2: ['growth', 'aggressive'],
    1: ['blend', 'core', 'balanced', 'world', 'global', 'international',
        'emerging', 'europe', 'asia', 'pacific', 'dividend'],
}

def _cap_from_market_cap(mc):
    if mc is None:
        return None
    if mc >= LARGE_CAP_THRESHOLD:
        return 0
    if mc >= MID_CAP_THRESHOLD:
        return 1
    return 2

def _style_from_ratios(pe, pb):
    """Very rough heuristic: low P/E & P/B → Value, high → Growth, else Blend."""
    score = 0
    count = 0
    if pe and pe > 0:
        score += pe
        count += 1
    if pb and pb > 0:
        score += pb * 5   # scale P/B to be comparable to P/E
        count += 1
    if count == 0:
        return 1  # default Blend
    avg = score / count
    if avg < 12:
        return 0  # Value
    if avg > 25:
        return 2  # Growth
    return 1      # Blend

def _search_keywords(text, keyword_dict):
    """Return the key whose keyword list first matches in text (lower-cased)."""
    text = text.lower()
    for key, words in keyword_dict.items():
        for w in words:
            if w in text:
                return key
    return None

def classify_ticker(ticker):
    """
    Returns (cap_row, style_col) or None if the asset is not classifiable
    as an equity (e.g. forex, commodity).
    """
    info = yf.Ticker(ticker).info

    quote_type = info.get('quoteType', '').upper()

    # Skip non-equity asset types
    if quote_type in ('CURRENCY', 'FUTURE', 'OPTION', 'INDEX', 'CRYPTOCURRENCY'):
        return None

    # --- Try Morningstar category string first (works for many ETFs) ---
    category = info.get('category') or info.get('categoryName') or ''
    if category:
        cap   = _search_keywords(category, CAP_KEYWORDS)
        style = _search_keywords(category, STYLE_KEYWORDS)
        if cap is not None and style is not None:
            return (cap, style)

    # --- For individual stocks: use marketCap + valuation ratios ---
    if quote_type == 'EQUITY':
        mc  = info.get('marketCap')
        pe  = info.get('trailingPE') or info.get('forwardPE')
        pb  = info.get('priceToBook')
        cap   = _cap_from_market_cap(mc)
        style = _style_from_ratios(pe, pb)
        if cap is None:
            cap = 0  # default Large if unknown
        return (cap, style)

    # --- ETF / MUTUALFUND without category: try fund name / long name ---
    name = (info.get('longName') or info.get('shortName') or ticker)
    cap   = _search_keywords(name, CAP_KEYWORDS)
    style = _search_keywords(name, STYLE_KEYWORDS)
    cap   = cap   if cap   is not None else 0  # default Large
    style = style if style is not None else 1  # default Blend
    return (cap, style)

print('Classifier ready.')

In [ ]:
# ---------------------------------------------------------------------------
# 3. Classify every holding and accumulate weighted allocations
# ---------------------------------------------------------------------------

# 3×3 grid: rows = [Large, Mid, Small], cols = [Value, Blend, Growth]
raw_grid = np.zeros((3, 3))   # stores sum of weights (0-1 scale)
skipped  = {}                 # non-equity assets
classified = {}

for ticker, weight in portfolio.items():
    print(f'Fetching {ticker} ...', end=' ')
    result = classify_ticker(ticker)
    if result is None:
        skipped[ticker] = weight
        print('skipped (non-equity)')
    else:
        cap_row, style_col = result
        raw_grid[cap_row, style_col] += weight
        classified[ticker] = {'weight': weight, 'cap': cap_row, 'style': style_col}
        cap_labels   = ['Large', 'Mid', 'Small']
        style_labels = ['Value', 'Blend', 'Growth']
        print(f'{cap_labels[cap_row]}-{style_labels[style_col]}')

equity_weight = raw_grid.sum()
print(f'\nTotal equity weight : {equity_weight*100:.2f}%')
if skipped:
    print('Skipped (non-equity):')
    for t, w in skipped.items():
        print(f'  {t}: {w*100:.2f}%')

In [ ]:
# ---------------------------------------------------------------------------
# 4. Normalise to percentage of equity portion (sums to 100)
# ---------------------------------------------------------------------------

if equity_weight == 0:
    raise ValueError('No equity holdings found – cannot draw a style box.')

allocations = (raw_grid / equity_weight) * 100   # each cell = % of equity

print('Style box allocations (% of equity portion):')
print(f'  {"":10s}  {"Value":>8s}  {"Blend":>8s}  {"Growth":>8s}')
for i, cap in enumerate(['Large', 'Mid', 'Small']):
    row = allocations[i]
    print(f'  {cap:10s}  {row[0]:8.2f}  {row[1]:8.2f}  {row[2]:8.2f}')
print(f'  Total: {allocations.sum():.2f}%')

In [ ]:
# ---------------------------------------------------------------------------
# 5. Plot the Equity Style Box
# ---------------------------------------------------------------------------

def plot_equity_style_box(allocations, categories, sizes):
    colors = {
        '30+':  '#1f77b4',
        '20-29':'#6baed6',
        '10-19':'#bdd7e7',
        '5-9':  '#eff3ff',
        '0-4':  '#f7fbff'
    }

    def get_color(value):
        if value >= 30: return colors['30+']
        if value >= 20: return colors['20-29']
        if value >= 10: return colors['10-19']
        if value >= 5:  return colors['5-9']
        return colors['0-4']

    fig, ax = plt.subplots(figsize=(4, 3))

    for i in range(allocations.shape[0]):
        for j in range(allocations.shape[1]):
            value = allocations[i, j]
            color = get_color(value)
            ax.add_patch(plt.Rectangle((j, 2 - i), 1, 1, fill=True, color=color))
            ax.text(j + 0.5, 2 - i + 0.5, f'{value:.1f}',
                    ha='center', va='center', fontsize=14,
                    color='black' if value < 10 else 'white')

    ax.set_xticks(np.arange(len(categories)) + 0.5)
    ax.set_yticks(np.arange(len(sizes)) + 0.5)
    ax.set_xticklabels(categories, fontsize=12)
    ax.set_yticklabels(sizes[::-1], fontsize=12, rotation=90, va='center')

    ax.set_xlim(0, len(categories))
    ax.set_ylim(0, len(sizes))
    ax.grid(False)
    ax.set_xticks(np.arange(len(categories)), minor=True)
    ax.set_yticks(np.arange(len(sizes)), minor=True)
    ax.grid(which='minor', color='black', linestyle='-', linewidth=2)
    ax.tick_params(which='major', bottom=False, left=False)

    legend_labels  = ['30+', '20-29', '10-19', '5-9', '0-4']
    legend_colors  = [colors[l] for l in legend_labels]
    legend_patches = [plt.Rectangle((0,0),1,1, color=c) for c in legend_colors]
    ax.legend(legend_patches, legend_labels, title='Weight %',
              loc='center left', bbox_to_anchor=(1, 0.5))

    plt.title('Equity Style Box\n(% of equity portion)', fontsize=14)
    plt.tight_layout()
    plt.show()


def print_sums(allocations, categories, sizes):
    row_sums = allocations.sum(axis=1)
    col_sums = allocations.sum(axis=0)

    print('Aggregate market-cap exposure:')
    for size, s in zip(sizes, row_sums):
        print(f'  {size}: {s:.1f}%')

    print('\nAggregate style exposure:')
    for cat, s in zip(categories, col_sums):
        print(f'  {cat}: {s:.1f}%')


categories = ['Value', 'Blend', 'Growth']
sizes      = ['Large', 'Mid',   'Small']

plot_equity_style_box(allocations, categories, sizes)
print_sums(allocations, categories, sizes)